# LVHM experiments for Part C

Reinforcement learning for adaptive job sequencing in a low-volume high-mix SO-ARM101 work cell (UTS 43008, Assignment 3, team Amazing Abundance).

This notebook is self-contained. It holds the whole environment, the six dispatch rules, masked PPO, masked double dueling DQN and the Part C evaluation, and needs nothing else from the repository.

It produces:
- learning curves for PPO and DQN (`results/learning_curves.png`)
- results tables for the four Part B test conditions (`results/partc_tables.md`)


## How to run on Amazon SageMaker

1. Upload this file (`experiments.ipynb`) to SageMaker (JupyterLab: Upload Files).
2. Use an instance with an NVIDIA T4 GPU, for example `ml.g4dn.xlarge`, and a PyTorch kernel.
3. Run all cells from top to bottom. The first cell installs the Python packages.

Training (sections 13 and 14) is the only slow part. Checkpoints are saved under `runs/`, and a checkpoint that already exists is reused when the notebook runs again.


## 1. Install packages


In [ ]:
# Packages this notebook needs.
# torch comes with the SageMaker PyTorch kernel (CUDA build). If your kernel has no torch, add torch to this line.
%pip install -q gymnasium sb3-contrib pandas matplotlib


## 2. Imports and device


In [ ]:
import time
from dataclasses import dataclass, replace
from pathlib import Path

import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from gymnasium import spaces
from sb3_contrib import MaskablePPO
from sb3_contrib.common.maskable.policies import MaskableActorCriticPolicy
from stable_baselines3.common.callbacks import BaseCallback
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.vec_env import DummyVecEnv
from torch import nn

# Use the T4 GPU when there is one
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE, torch.cuda.get_device_name(0) if DEVICE == 'cuda' else '')


## 3. Settings

Every experiment setting lives in this cell. Each algorithm trains once per seed.


In [ ]:
SEEDS = [0, 1, 2]          # one training run per seed
PPO_TIMESTEPS = 300_000    # Part B plans 1M steps; reduced for the initial results
DQN_TIMESTEPS = 300_000
VAL_EVERY = 25_000         # score the policy on the validation shifts this often
N_VAL_SHIFTS = 20          # validation shifts choose the best checkpoint
N_TEST_SHIFTS = 100        # test shifts per condition
VAL_SEED, TEST_SEED = 20_000, 10_000  # validation and test shifts never overlap
RETRAIN = False            # False: reuse checkpoints that already exist

RUN_DIR = Path('runs')         # checkpoints and learning curves
RESULTS_DIR = Path('results')  # tables and figures for the report
RESULTS_DIR.mkdir(exist_ok=True)


## 4. Products and environment settings

- Time is in seconds; one shift is an 8-minute run of the cell (480 s).
- Motion timing was measured in MuJoCo with the SO-ARM101 model (see the repository README).
- A product is only its weight and fragility. Heavy or fragile items are moved more slowly, so processing time follows from those two features.
- Careful mode slows the loaded motion further (x1.5) and cuts the drop probability (x0.3).


In [ ]:
# Products: (weight in kg, fragility from 0 to 1). The agent never sees the product name.
PRODUCTS = {
    'P1': (0.03, 0.10),  # light, robust
    'P2': (0.08, 0.60),  # light, fragile
    'P3': (0.18, 0.20),  # heavy, robust
    'P4': (0.05, 0.85),  # light, very fragile
    'P5': (0.12, 0.45),  # never trained on, inside the training range
    'P6': (0.23, 0.95),  # never trained on, outside the training range
}
PRODUCT_NAMES = tuple(PRODUCTS)
FAMILIAR = ('P1', 'P2', 'P3', 'P4')

# Penalty weights on test shifts: alpha per late second, beta per switch second, delta per drop
EVAL_WEIGHTS = (0.015, 0.15, 0.6)


@dataclass(frozen=True)
class EnvConfig:
    # Shift: jobs arrive as a Poisson process
    shift_length: float = 480.0
    arrival_rate: float = 0.14                # jobs per second on test shifts (0.28 = busy day)
    arrival_rate_range: tuple = (0.07, 0.21)  # training draws a rate per episode
    initial_jobs: int = 3                     # backlog waiting at the start of the shift
    slack: tuple = (5.0, 50.0)                # deadline = arrival + processing + slack

    # SO-ARM101 timing measured in MuJoCo; one station per tray, the arm stays at the last station
    num_trays: int = 2
    travel_time: float = 1.15                 # extra swing when switching station
    empty_time: float = 1.388                 # swing to the tray, descend, close the gripper
    loaded_time_base: float = 0.929           # loaded motion = base + per_slowdown * slowdown
    loaded_time_per_slowdown: float = 0.987

    # Handling speed: slowdown = 1 + 1.5 * weight / 0.25 + 1.5 * fragility
    slow_weight: float = 1.5
    slow_fragility: float = 1.5

    # Noise and drops
    proc_noise_sigma: float = 0.10            # log-normal noise on each attempt
    drop_base: float = 0.01
    drop_fragility: float = 0.20
    drop_weight: float = 0.05
    drop_max: float = 0.5
    drop_time_fraction: float = 0.5           # share of the processing time lost on a drop
    max_attempts: int = 2                     # the second drop scraps the job

    # Careful pick mode
    careful_mode: bool = True
    careful_drop_factor: float = 0.3
    careful_slowdown: float = 1.5

    # Fixed reward terms
    on_time_bonus: float = 1.0
    unfinished_penalty: float = 1.0

    # Observation: the 8 oldest waiting jobs and these scales
    num_slots: int = 8
    proc_scale: float = 8.0
    weight_scale: float = 0.25
    deadline_scale: float = 60.0
    wait_scale: float = 120.0
    max_queue_obs: int = 32

    # Training ranges: features (weight, fragility) and penalty weights (alpha, beta, delta)
    feature_low: tuple = (0.02, 0.0)
    feature_high: tuple = (0.20, 0.9)
    weight_low: tuple = (0.005, 0.05, 0.2)
    weight_high: tuple = (0.03, 0.30, 1.5)
    holdout_radius: float = 0.1               # box around P5 (normalised features) that training never samples

    def slowdown(self, weight, fragility):
        return 1.0 + self.slow_weight * np.asarray(weight) / self.weight_scale + self.slow_fragility * np.asarray(fragility)

    def proc_time(self, weight, fragility, careful=False):
        # Nominal time of one pick-and-place, before noise
        slow = self.slowdown(weight, fragility) * (self.careful_slowdown if careful else 1.0)
        return self.empty_time + self.loaded_time_base + self.loaded_time_per_slowdown * slow

    def drop_probability(self, weight, fragility, careful=False):
        q = self.drop_base + self.drop_fragility * np.asarray(fragility) + self.drop_weight * np.asarray(weight) / self.weight_scale
        q = np.minimum(q, self.drop_max)
        return q * self.careful_drop_factor if careful else q


ENV = EnvConfig()

# The products at a glance
pd.DataFrame(
    [
        {
            'product': name,
            'weight_kg': w,
            'fragility': f,
            'processing_s': round(float(ENV.proc_time(w, f)), 2),
            'drop_prob': round(float(ENV.drop_probability(w, f)), 3),
        }
        for name, (w, f) in PRODUCTS.items()
    ]
)


## 5. Shift instances

A shift instance draws everything random about one shift up front: arrivals, deadlines, processing noise and the random number behind every possible drop. Every policy evaluated on the same instance therefore faces the same jobs and the same drop outcome for the same pick (common random numbers).

Training samples weight and fragility continuously, but never near P5, so P5 is truly unseen.


In [ ]:
@dataclass(frozen=True)
class ShiftInstance:
    arrival: np.ndarray    # arrival time of each job, ascending
    proc: np.ndarray       # nominal processing time in normal mode
    weight: np.ndarray
    fragility: np.ndarray
    tray: np.ndarray       # source tray, 0 or 1
    deadline: np.ndarray
    noise: np.ndarray      # processing-time noise per job and attempt
    u_drop: np.ndarray     # uniform draw per job and attempt; the pick drops if it is below the drop probability

    def __len__(self):
        return len(self.arrival)


def normalise_features(cfg, feats):
    low, high = np.array(cfg.feature_low), np.array(cfg.feature_high)
    return (np.asarray(feats, dtype=float) - low) / (high - low)


def in_holdout(cfg, feats):
    # True inside the box around P5 that training never samples
    p5 = normalise_features(cfg, PRODUCTS['P5'])
    return np.abs(normalise_features(cfg, feats) - p5).max(axis=-1) < cfg.holdout_radius


def sample_training_features(cfg, rng, n):
    # Uniform (weight, fragility); anything that lands near P5 is drawn again
    low, high = np.array(cfg.feature_low), np.array(cfg.feature_high)
    feats = rng.uniform(low, high, size=(n, 2))
    reject = in_holdout(cfg, feats)
    while reject.any():
        feats[reject] = rng.uniform(low, high, size=(int(reject.sum()), 2))
        reject = in_holdout(cfg, feats)
    return feats


def sample_instance(cfg, rng, products=None, arrival_rate=None):
    # products=None: continuous training features; otherwise each job is one of the named products
    rate = cfg.arrival_rate if arrival_rate is None else arrival_rate
    n_arrivals = rng.poisson(rate * cfg.shift_length)
    arrival = np.concatenate([np.zeros(cfg.initial_jobs), np.sort(rng.uniform(0.0, cfg.shift_length, n_arrivals))])
    n = len(arrival)
    if products is None:
        feats = sample_training_features(cfg, rng, n)
    else:
        product = np.array([PRODUCT_NAMES.index(p) for p in products])[rng.integers(len(products), size=n)]
        feats = np.array([PRODUCTS[PRODUCT_NAMES[i]] for i in product]).reshape(n, 2)
    weight, fragility = feats[:, 0], feats[:, 1]
    proc = cfg.proc_time(weight, fragility)
    attempts = (n, cfg.max_attempts)
    return ShiftInstance(
        arrival=arrival,
        proc=proc,
        weight=weight,
        fragility=fragility,
        tray=rng.integers(cfg.num_trays, size=n),
        deadline=arrival + proc + rng.uniform(*cfg.slack, size=n),
        noise=np.exp(rng.normal(0.0, cfg.proc_noise_sigma, size=attempts)),
        u_drop=rng.uniform(size=attempts),
    )


def make_instances(cfg, products, n, seed):
    # A fixed, reproducible set of shifts
    return [sample_instance(cfg, np.random.default_rng([seed, i]), products) for i in range(n)]


# Check the hold-out: training never samples near P5, and P6 lies outside the training range
assert not in_holdout(ENV, sample_training_features(ENV, np.random.default_rng(0), 100_000)).any()
assert in_holdout(ENV, PRODUCTS['P5'])
assert np.any((np.array(PRODUCTS['P6']) < ENV.feature_low) | (np.array(PRODUCTS['P6']) > ENV.feature_high))


## 6. The environment

A discrete-event Gymnasium environment. A decision happens whenever the arm is idle and jobs are waiting; one step simulates the whole pick-and-place.

- **Observation (79 values):** the 8 oldest waiting jobs (present, processing time, weight, fragility, time to deadline, tray, waiting time, dropped once) and global features (shift remaining, arm station, queue length, penalty weights).
- **Action (16):** `a % 8` is the job slot, `a >= 8` means careful mode. Empty slots are masked.
- **Reward:** +1 per job finished on time, minus alpha per late second (charged as lateness accrues for every waiting job), minus beta per switch second, minus delta per drop, minus 1 per job still waiting at the end of the shift.
- The penalty weights are sampled per training episode and shown to the agent, so they can change at test time without retraining.


In [ ]:
class LVHMSchedulingEnv(gym.Env):
    def __init__(self, config=None):
        super().__init__()
        self.cfg = cfg = config or EnvConfig()
        self.num_modes = 2 if cfg.careful_mode else 1
        # per slot: present, proc, weight, fragility, time to deadline, tray one-hot, waiting, dropped once
        self.slot_dim = 7 + cfg.num_trays
        # global: shift remaining, arm station one-hot, queue length, penalty weights
        self.global_dim = 1 + cfg.num_trays + 1 + 3
        self.action_space = spaces.Discrete(cfg.num_slots * self.num_modes)

        slot_low = np.zeros(self.slot_dim, dtype=np.float32)
        slot_low[4] = -1.0  # time to deadline is negative once a job is late
        global_high = np.ones(self.global_dim, dtype=np.float32)
        global_high[1 + cfg.num_trays] = cfg.max_queue_obs / cfg.num_slots
        global_high[-3:] = 2.0  # test-time weights may exceed the training range
        low = np.concatenate([np.tile(slot_low, cfg.num_slots), np.zeros(self.global_dim, dtype=np.float32)])
        high = np.concatenate([np.ones(self.slot_dim * cfg.num_slots, dtype=np.float32), global_high])
        self.observation_space = spaces.Box(low, high, dtype=np.float32)
        self._inst = None

    def reset(self, *, seed=None, options=None):
        # options: instance (a fixed shift), products, arrival_rate, weights (alpha, beta, delta)
        super().reset(seed=seed)
        options = options or {}
        cfg = self.cfg
        inst = options.get('instance')
        if inst is None:
            rate = options.get('arrival_rate')
            if rate is None and cfg.arrival_rate_range is not None:
                rate = self.np_random.uniform(*cfg.arrival_rate_range)
            inst = sample_instance(cfg, self.np_random, options.get('products'), rate)
        weights = options.get('weights')
        if weights is None:
            weights = self.np_random.uniform(cfg.weight_low, cfg.weight_high)

        self._inst = inst
        self._w = np.asarray(weights, dtype=float)
        self._attempts = np.zeros(len(inst), dtype=int)
        self._queue = []   # waiting job indices, in arrival order
        self._next = 0     # index of the next job to arrive
        self._t = 0.0
        self._arm = 0      # station the arm is at
        self._stats = {k: 0 for k in ('decisions', 'completed', 'on_time', 'late', 'drops', 'scrapped', 'unfinished', 'careful')}
        self._stats.update(tardiness=0.0, travel=0.0, score=0.0)

        self._admit(self._t)
        if not self._queue:
            self._t = float(inst.arrival[self._next])
            self._admit(self._t)
        return self._obs(), {'t': self._t}

    def step(self, action):
        cfg, inst = self.cfg, self._inst
        alpha, beta, delta = self._w
        if not self.action_masks()[action]:
            raise ValueError(f'action {action} selects an empty slot')
        slot, careful = int(action) % cfg.num_slots, int(action) // cfg.num_slots == 1
        j = self._queue[slot]

        # Simulate the pick: a station switch if needed, then the pick-and-place or a drop
        t0 = self._t
        travel = cfg.travel_time if inst.tray[j] != self._arm else 0.0
        attempt = self._attempts[j]
        proc = cfg.proc_time(inst.weight[j], inst.fragility[j], careful) * inst.noise[j, attempt]
        dropped = bool(inst.u_drop[j, attempt] < cfg.drop_probability(inst.weight[j], inst.fragility[j], careful))
        t1 = t0 + travel + (cfg.drop_time_fraction * proc if dropped else proc)

        # Lateness accrued during [t0, t1] by every job in the system, including jobs that arrived meanwhile
        self._admit(t1)
        deadlines = inst.deadline[self._queue]
        late = float(np.maximum(0.0, t1 - np.maximum(t0, deadlines)).sum())

        reward = -alpha * late - beta * travel
        self._arm = int(inst.tray[j])
        if dropped:
            self._attempts[j] += 1
            self._stats['drops'] += 1
            reward -= delta
            if self._attempts[j] >= cfg.max_attempts:  # second drop: scrap the job
                self._queue.remove(j)
                self._stats['scrapped'] += 1
        else:
            self._queue.remove(j)
            self._stats['completed'] += 1
            if t1 <= inst.deadline[j]:
                self._stats['on_time'] += 1
                reward += cfg.on_time_bonus
            else:
                self._stats['late'] += 1

        self._t = t1
        terminated = self._advance()
        if terminated:
            self._stats['unfinished'] = len(self._queue)
            reward -= cfg.unfinished_penalty * len(self._queue)

        self._stats['decisions'] += 1
        self._stats['careful'] += careful
        self._stats['tardiness'] += late
        self._stats['travel'] += travel
        self._stats['score'] += reward

        info = {'t': self._t, 'job': j, 'slot': slot, 'careful': careful, 'start': t0, 'end': t1, 'travel': travel, 'dropped': dropped}
        if terminated:
            info['episode_metrics'] = dict(self._stats)
        return self._obs(), float(reward), terminated, False, info

    def action_masks(self):
        # Valid actions (MaskablePPO looks for this method): occupied slots, in both modes
        slots = np.zeros(self.cfg.num_slots, dtype=bool)
        slots[: min(len(self._queue), self.cfg.num_slots)] = True
        return np.tile(slots, self.num_modes)

    def _admit(self, t):
        # Jobs that have arrived by time t join the queue
        while self._next < len(self._inst) and self._inst.arrival[self._next] <= t:
            self._queue.append(self._next)
            self._next += 1

    def _advance(self):
        # Move the clock to the next decision point; True when the shift is over
        if self._t >= self.cfg.shift_length:
            return True
        if self._queue:
            return False
        if self._next >= len(self._inst) or self._inst.arrival[self._next] >= self.cfg.shift_length:
            return True
        self._t = float(self._inst.arrival[self._next])
        self._admit(self._t)
        return False

    def _obs(self):
        cfg, inst, t = self.cfg, self._inst, self._t
        vis = np.array(self._queue[: cfg.num_slots], dtype=int)
        n = len(vis)
        slots = np.zeros((cfg.num_slots, self.slot_dim), dtype=np.float32)
        slots[:n, 0] = 1.0                                                                # present
        slots[:n, 1] = np.minimum(inst.proc[vis] / cfg.proc_scale, 1.0)                   # processing time
        slots[:n, 2] = np.minimum(inst.weight[vis] / cfg.weight_scale, 1.0)               # weight
        slots[:n, 3] = inst.fragility[vis]                                                # fragility
        slots[:n, 4] = np.clip((inst.deadline[vis] - t) / cfg.deadline_scale, -1.0, 1.0)  # time to deadline
        slots[np.arange(n), 5 + inst.tray[vis]] = 1.0                                     # tray one-hot
        slots[:n, 5 + cfg.num_trays] = np.minimum((t - inst.arrival[vis]) / cfg.wait_scale, 1.0)  # waiting time
        slots[:n, 6 + cfg.num_trays] = self._attempts[vis] > 0                            # dropped once

        w_low, w_high = np.array(cfg.weight_low), np.array(cfg.weight_high)
        glob = np.concatenate(
            [
                [max(0.0, (cfg.shift_length - t) / cfg.shift_length)],           # shift remaining
                np.eye(cfg.num_trays)[self._arm],                                # arm station
                [min(len(self._queue), cfg.max_queue_obs) / cfg.num_slots],      # queue length
                np.clip((self._w - w_low) / (w_high - w_low), 0.0, 2.0),         # penalty weights
            ]
        ).astype(np.float32)
        return np.concatenate([slots.ravel(), glob])


## 7. Rule-based baselines

The six dispatch rules from Part B. Each sees the same 8 jobs as the agent, always picks in normal mode, and breaks ties by earliest arrival.


In [ ]:
def rule(key):
    # A policy that serves the visible job with the smallest key; policies take (obs, env) and return an action
    def policy(obs, env):
        vis = np.array(env._queue[: env.cfg.num_slots], dtype=int)
        return int(np.argmin(key(env._inst, vis, env)))

    return policy


fcfs = rule(lambda inst, vis, env: inst.arrival[vis])                          # first come, first served
spt = rule(lambda inst, vis, env: inst.proc[vis])                              # shortest processing time
lrf = rule(lambda inst, vis, env: inst.fragility[vis])                         # lowest failure risk first
edf = rule(lambda inst, vis, env: inst.deadline[vis])                          # earliest deadline first
ntf = rule(lambda inst, vis, env: (inst.tray[vis] != env._arm).astype(float))  # nearest tray first


class RandomPolicy:
    def __init__(self, seed=0):
        self.rng = np.random.default_rng(seed)

    def __call__(self, obs, env):
        valid = np.flatnonzero(env.action_masks()[: env.cfg.num_slots])
        return int(self.rng.choice(valid))


def rule_baselines(seed=0):
    return {'FCFS': fcfs, 'SPT': spt, 'LRF': lrf, 'EDF': edf, 'NTF': ntf, 'Random': RandomPolicy(seed)}


## 8. Evaluation helpers

`evaluate` runs a policy on fixed shifts with fixed penalty weights. `Validator` scores a policy on the validation shifts during training, keeps the best checkpoint and logs the learning curve.


In [ ]:
METRICS = ['on_time', 'tardiness', 'travel', 'drops', 'scrapped', 'unfinished', 'careful', 'score']


def run_episode(env, policy, instance, weights=EVAL_WEIGHTS):
    obs, _ = env.reset(options={'instance': instance, 'weights': weights})
    while True:
        obs, _, terminated, truncated, info = env.step(policy(obs, env))
        if terminated or truncated:
            return info['episode_metrics']


def evaluate(policy, instances, cfg=ENV, weights=EVAL_WEIGHTS):
    env = LVHMSchedulingEnv(cfg)
    return [run_episode(env, policy, inst, weights) for inst in instances]


def mean_metrics(results):
    return {m: float(np.mean([r[m] for r in results])) for m in METRICS}


class Validator:
    def __init__(self, out_dir):
        self.instances = make_instances(ENV, FAMILIAR, N_VAL_SHIFTS, VAL_SEED)
        self.out_dir = out_dir
        self.best = -np.inf
        self.rows = []

    def __call__(self, policy, timesteps, save_best):
        row = {'timesteps': timesteps} | mean_metrics(evaluate(policy, self.instances))
        self.rows.append(row)
        if row['score'] > self.best:
            self.best = row['score']
            save_best()
        pd.DataFrame(self.rows).to_csv(self.out_dir / 'validation.csv', index=False)
        print(f'  step {timesteps:>7}  validation score', round(row['score'], 2))


## 9. Quick check: rules on the test shifts

A fast sanity run (seconds). These numbers match the baseline tables in the repository README.


In [ ]:
test_shifts = make_instances(ENV, FAMILIAR, N_TEST_SHIFTS, TEST_SEED)
pd.DataFrame({name: mean_metrics(evaluate(p, test_shifts)) for name, p in rule_baselines().items()}).T.round(2)


## 10. Networks shared by PPO and DQN

Every waiting job is embedded by the same MLP from its own features plus the global features. The embeddings are pooled (mean and max over real jobs) into a queue context, so a job's score can depend on the rest of the queue. Scores come out per (job slot, pick mode), and padded slots never influence them.


In [ ]:
@dataclass(frozen=True)
class ObsLayout:
    num_slots: int
    slot_dim: int
    global_dim: int
    num_modes: int

    @property
    def num_actions(self):
        return self.num_slots * self.num_modes


def layout_of(env):
    return ObsLayout(env.cfg.num_slots, env.slot_dim, env.global_dim, env.num_modes)


def mlp(sizes):
    # Linear layers with ReLU in between (none after the last one)
    layers = []
    for i, (a, b) in enumerate(zip(sizes, sizes[1:])):
        layers.append(nn.Linear(a, b))
        if i < len(sizes) - 2:
            layers.append(nn.ReLU())
    return nn.Sequential(*layers)


class QueueEncoder(nn.Module):
    def __init__(self, layout, hidden=64):
        super().__init__()
        self.layout = layout
        self.job = nn.Sequential(mlp([layout.slot_dim + layout.global_dim, hidden, hidden]), nn.ReLU())
        self.context_dim = 2 * hidden

    def forward(self, obs):
        # returns job embeddings (batch, K, hidden), queue context (batch, 2 * hidden), global features
        k, s = self.layout.num_slots, self.layout.slot_dim
        slots = obs[:, : k * s].reshape(-1, k, s)
        glob = obs[:, k * s :]
        present = slots[..., 0] > 0.5
        h = self.job(torch.cat([slots, glob.unsqueeze(1).expand(-1, k, -1)], dim=-1))
        mask = present.unsqueeze(-1)
        mean = (h * mask).sum(1) / mask.sum(1).clamp(min=1)
        mx = h.masked_fill(~mask, float('-inf')).max(1).values
        mx = torch.where(present.any(1, keepdim=True), mx, torch.zeros_like(mx))  # empty queue: zeros
        return h, torch.cat([mean, mx], dim=-1), glob


class JobScorer(nn.Module):
    # One score per (job slot, pick mode), laid out as action = mode * K + slot
    def __init__(self, layout, embed_dim, context_dim, hidden=64):
        super().__init__()
        self.layout = layout
        self.net = mlp([embed_dim + context_dim, hidden, layout.num_modes])

    def forward(self, h, context):
        k = self.layout.num_slots
        scores = self.net(torch.cat([h, context.unsqueeze(1).expand(-1, k, -1)], dim=-1))  # (batch, K, modes)
        return scores.transpose(1, 2).reshape(h.shape[0], -1)


class ValueHead(nn.Module):
    def __init__(self, context_dim, global_dim, hidden=64):
        super().__init__()
        self.net = mlp([context_dim + global_dim, hidden, 1])

    def forward(self, context, glob):
        return self.net(torch.cat([context, glob], dim=-1))


## 11. Masked PPO (sb3-contrib)

sb3-contrib `MaskablePPO` handles action masking. The job scorer replaces SB3's default policy network: its scores are the action logits, and the value function reads the pooled queue context.


In [ ]:
PPO_PARAMS = dict(
    learning_rate=3e-4,
    n_steps=256,      # rollout length per env; 8 envs give 2048 transitions per update
    batch_size=256,
    n_epochs=10,
    gamma=0.99,
    gae_lambda=0.95,
    ent_coef=0.01,    # entropy bonus, the exploration knob (DQN uses epsilon instead)
)


class JobScoringExtractor(nn.Module):
    # Fills SB3's mlp_extractor slot: latent_pi is already the action logits, latent_vf a queue embedding
    def __init__(self, layout, hidden):
        super().__init__()
        self.encoder = QueueEncoder(layout, hidden)
        self.scorer = JobScorer(layout, hidden, self.encoder.context_dim, hidden)
        self.value = nn.Sequential(mlp([self.encoder.context_dim + layout.global_dim, hidden, hidden]), nn.ReLU())
        self.latent_dim_pi = layout.num_actions
        self.latent_dim_vf = hidden

    def forward(self, features):
        h, context, glob = self.encoder(features)
        return self.scorer(h, context), self.value(torch.cat([context, glob], dim=-1))

    def forward_actor(self, features):
        h, context, _ = self.encoder(features)
        return self.scorer(h, context)

    def forward_critic(self, features):
        _, context, glob = self.encoder(features)
        return self.value(torch.cat([context, glob], dim=-1))


class JobScoringPolicy(MaskableActorCriticPolicy):
    def __init__(self, *args, layout, hidden=64, **kwargs):
        self.layout, self.hidden = layout, hidden
        super().__init__(*args, **kwargs)

    def _build_mlp_extractor(self):
        self.mlp_extractor = JobScoringExtractor(self.layout, self.hidden)

    def _build(self, lr_schedule):
        super()._build(lr_schedule)
        # The scorer already outputs logits: drop SB3's extra action layer and start the logits small
        self.action_net = nn.Identity()
        nn.init.orthogonal_(self.mlp_extractor.scorer.net[-1].weight, gain=0.01)
        nn.init.zeros_(self.mlp_extractor.scorer.net[-1].bias)
        self.optimizer = self.optimizer_class(self.parameters(), lr=lr_schedule(1), **self.optimizer_kwargs)

    def _get_constructor_parameters(self):
        return super()._get_constructor_parameters() | {'layout': self.layout, 'hidden': self.hidden}


def ppo_policy(model):
    # Deterministic masked policy for evaluation
    def policy(obs, env):
        action, _ = model.predict(obs, action_masks=env.action_masks(), deterministic=True)
        return int(action)

    return policy


class ValidationCallback(BaseCallback):
    # Runs the validator every VAL_EVERY steps and at the end of training
    def __init__(self, validator):
        super().__init__()
        self.validator, self.next_eval = validator, 0

    def _on_step(self):
        if self.num_timesteps >= self.next_eval:
            self.next_eval += VAL_EVERY
            self._validate()
        return True

    def _on_training_end(self):
        self._validate()

    def _validate(self):
        save = lambda: self.model.save(self.validator.out_dir / 'best_model')
        self.validator(ppo_policy(self.model), self.num_timesteps, save)


def train_ppo(seed, timesteps, out_dir, device=DEVICE, n_envs=8, hidden=64):
    out_dir.mkdir(parents=True, exist_ok=True)
    venv = make_vec_env(LVHMSchedulingEnv, n_envs, seed=seed, env_kwargs={'config': ENV}, vec_env_cls=DummyVecEnv)
    layout = layout_of(LVHMSchedulingEnv(ENV))
    model = MaskablePPO(JobScoringPolicy, venv, policy_kwargs={'layout': layout, 'hidden': hidden}, seed=seed, device=device, **PPO_PARAMS)
    validator = Validator(out_dir)
    start = time.perf_counter()
    model.learn(timesteps, callback=ValidationCallback(validator))
    return {'seed': seed, 'device': str(model.device), 'minutes': round((time.perf_counter() - start) / 60, 1), 'best_validation_score': validator.best}


def load_ppo(path):
    # Evaluation runs one shift at a time, which is faster on the CPU
    return ppo_policy(MaskablePPO.load(path, device='cpu'))


## 12. Masked double dueling DQN (PyTorch)

SB3's `DQN` has no action masking, no double-DQN target and no dueling head, so this agent is written directly in PyTorch.

- Dueling: Q(s, a) = V(s) + A(s, a) - mean of A over valid actions. A comes from the job scorer, V from the pooled queue context.
- Double DQN: the online network picks the next action, the target network scores it.
- Masking: invalid actions get a huge negative Q, so they are never chosen and never used in a target.
- Exploration: epsilon decays linearly from 1.0 to 0.05 over the first 30% of training.


In [ ]:
@dataclass
class DQNConfig:
    timesteps: int = DQN_TIMESTEPS
    n_envs: int = 8
    lr: float = 1e-4
    gamma: float = 0.99
    batch_size: int = 128
    buffer_size: int = 100_000
    learning_starts: int = 5_000
    train_freq: int = 4           # environment steps per gradient update
    tau: float = 0.005            # soft update rate of the target network
    eps_start: float = 1.0
    eps_end: float = 0.05
    eps_fraction: float = 0.3     # share of training over which epsilon decays
    warmup_edf_steps: int = 0     # optionally act with EDF first to seed the replay buffer
    hidden: int = 64
    grad_clip: float = 10.0
    device: str = DEVICE


class DuelingJobQNet(nn.Module):
    def __init__(self, layout, hidden=64):
        super().__init__()
        self.encoder = QueueEncoder(layout, hidden)
        self.advantage = JobScorer(layout, hidden, self.encoder.context_dim, hidden)
        self.value = ValueHead(self.encoder.context_dim, layout.global_dim, hidden)

    def forward(self, obs, mask):
        h, context, glob = self.encoder(obs)
        adv = self.advantage(h, context)
        adv_mean = (adv * mask).sum(1, keepdim=True) / mask.sum(1, keepdim=True).clamp(min=1)
        q = self.value(context, glob) + adv - adv_mean
        return q.masked_fill(~mask, -1e9)


class ReplayBuffer:
    def __init__(self, capacity, obs_dim, num_actions):
        self.capacity, self.size, self.pos = capacity, 0, 0
        self.obs = np.zeros((capacity, obs_dim), dtype=np.float32)
        self.next_obs = np.zeros((capacity, obs_dim), dtype=np.float32)
        self.mask = np.zeros((capacity, num_actions), dtype=bool)
        self.next_mask = np.zeros((capacity, num_actions), dtype=bool)
        self.action = np.zeros(capacity, dtype=np.int64)
        self.reward = np.zeros(capacity, dtype=np.float32)
        self.done = np.zeros(capacity, dtype=np.float32)

    def add(self, obs, mask, action, reward, next_obs, next_mask, done):
        i = self.pos
        self.obs[i], self.mask[i], self.action[i], self.reward[i] = obs, mask, action, reward
        self.next_obs[i], self.next_mask[i], self.done[i] = next_obs, next_mask, done
        self.pos = (self.pos + 1) % self.capacity
        self.size = min(self.size + 1, self.capacity)

    def sample(self, rng, batch_size, device):
        idx = rng.integers(self.size, size=batch_size)
        names = ('obs', 'mask', 'action', 'reward', 'next_obs', 'next_mask', 'done')
        return {name: torch.as_tensor(getattr(self, name)[idx], device=device) for name in names}


def dqn_policy(net):
    # Greedy masked policy for evaluation
    device = next(net.parameters()).device

    def policy(obs, env):
        with torch.no_grad():
            q = net(torch.as_tensor(obs[None], device=device), torch.as_tensor(env.action_masks()[None], device=device))
        return int(q.argmax(1))

    return policy


def dqn_update(online, target, optimizer, batch, cfg):
    q = online(batch['obs'], batch['mask']).gather(1, batch['action'][:, None]).squeeze(1)
    with torch.no_grad():
        best_next = online(batch['next_obs'], batch['next_mask']).argmax(1, keepdim=True)  # online net picks
        next_q = target(batch['next_obs'], batch['next_mask']).gather(1, best_next).squeeze(1)  # target net scores
        done = batch['done'].bool()
        next_q = torch.where(done, torch.zeros_like(next_q), next_q)  # no future after the shift ends
        y = batch['reward'] + cfg.gamma * next_q
    loss = F.smooth_l1_loss(q, y)
    optimizer.zero_grad()
    loss.backward()
    nn.utils.clip_grad_norm_(online.parameters(), cfg.grad_clip)
    optimizer.step()
    with torch.no_grad():
        for p, tp in zip(online.parameters(), target.parameters()):
            tp.lerp_(p, cfg.tau)


def train_dqn(seed, cfg, out_dir):
    out_dir.mkdir(parents=True, exist_ok=True)
    torch.manual_seed(seed)
    rng = np.random.default_rng(seed)
    device = torch.device(cfg.device)
    envs = [LVHMSchedulingEnv(ENV) for _ in range(cfg.n_envs)]
    layout = layout_of(envs[0])
    online = DuelingJobQNet(layout, cfg.hidden).to(device)
    target = DuelingJobQNet(layout, cfg.hidden).to(device)
    target.load_state_dict(online.state_dict())
    target.requires_grad_(False)
    optimizer = torch.optim.Adam(online.parameters(), lr=cfg.lr)
    buffer = ReplayBuffer(cfg.buffer_size, envs[0].observation_space.shape[0], layout.num_actions)
    validator = Validator(out_dir)

    def validate(step):
        save = lambda: torch.save(online.state_dict(), out_dir / 'best_model.pt')
        validator(dqn_policy(online), step, save)

    obs = np.stack([env.reset(seed=seed * 1000 + i)[0] for i, env in enumerate(envs)])
    masks = np.stack([env.action_masks() for env in envs])
    step, next_val, updates_due = 0, 0, 0.0
    start = time.perf_counter()

    while step < cfg.timesteps:
        if step >= next_val:
            validate(step)
            next_val += VAL_EVERY

        # Act in every env: epsilon-greedy over valid actions
        eps = max(cfg.eps_end, cfg.eps_start - (cfg.eps_start - cfg.eps_end) * step / (cfg.eps_fraction * cfg.timesteps))
        with torch.no_grad():
            greedy = online(torch.as_tensor(obs, device=device), torch.as_tensor(masks, device=device)).argmax(1).cpu().numpy()
        for i, env in enumerate(envs):
            if step < cfg.warmup_edf_steps:
                action = edf(obs[i], env)
            elif rng.random() < eps:
                action = int(rng.choice(np.flatnonzero(masks[i])))
            else:
                action = int(greedy[i])
            next_obs, reward, terminated, truncated, _ = env.step(action)
            next_mask = env.action_masks()
            buffer.add(obs[i], masks[i], action, reward, next_obs, next_mask, float(terminated))
            if terminated or truncated:
                next_obs, _ = env.reset()
                next_mask = env.action_masks()
            obs[i], masks[i] = next_obs, next_mask
        step += cfg.n_envs

        # Learn: one gradient update every train_freq environment steps
        if step < cfg.learning_starts:
            continue
        updates_due += cfg.n_envs / cfg.train_freq
        while updates_due >= 1:
            updates_due -= 1
            dqn_update(online, target, optimizer, buffer.sample(rng, cfg.batch_size, device), cfg)

    validate(step)
    return {'seed': seed, 'device': str(device), 'minutes': round((time.perf_counter() - start) / 60, 1), 'best_validation_score': validator.best}


def load_dqn(path, hidden=64):
    net = DuelingJobQNet(layout_of(LVHMSchedulingEnv(ENV)), hidden)
    net.load_state_dict(torch.load(path, map_location='cpu'))
    return dqn_policy(net.eval())


## 13. Train masked PPO

One run per seed. Each run prints its validation score every 25k steps and keeps the best checkpoint as `runs/ppo/seed<N>/best_model.zip`. SB3 may warn that PPO is meant for the CPU: the network is small, so the GPU gains little here, and `device='cpu'` is fine too.


In [ ]:
for seed in SEEDS:
    out = RUN_DIR / 'ppo' / f'seed{seed}'
    if (out / 'best_model.zip').exists() and not RETRAIN:
        print(f'PPO seed {seed}: checkpoint found, skipping training')
        continue
    print(f'PPO seed {seed}')
    print(train_ppo(seed, PPO_TIMESTEPS, out))


## 14. Train masked double dueling DQN

One run per seed; the best checkpoint is kept as `runs/dqn/seed<N>/best_model.pt`.


In [ ]:
dqn_config = DQNConfig()  # change hyperparameters here, e.g. DQNConfig(lr=5e-5)

for seed in SEEDS:
    out = RUN_DIR / 'dqn' / f'seed{seed}'
    if (out / 'best_model.pt').exists() and not RETRAIN:
        print(f'DQN seed {seed}: checkpoint found, skipping training')
        continue
    print(f'DQN seed {seed}')
    print(train_dqn(seed, dqn_config, out))


## 15. Learning curves

Validation score against environment steps, one line per seed.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, algo in zip(axes, ['ppo', 'dqn']):
    for seed in SEEDS:
        curve_file = RUN_DIR / algo / f'seed{seed}' / 'validation.csv'
        if curve_file.exists():
            curve = pd.read_csv(curve_file)
            ax.plot(curve['timesteps'], curve['score'], label=f'seed {seed}')
    ax.set_title(algo.upper())
    ax.set_xlabel('environment steps')
    ax.legend()
axes[0].set_ylabel('validation score')
fig.savefig(RESULTS_DIR / 'learning_curves.png', dpi=150, bbox_inches='tight')
plt.show()


## 16. Evaluate every policy on the test conditions

The four Part B tests (test 2 split into P5 and P6). Unseen products join the familiar mix, so P5 or P6 makes up a fifth of the jobs. Every policy faces the same 100 shifts per condition; each learned policy is evaluated once per seed.


In [ ]:
@dataclass(frozen=True)
class Condition:
    name: str
    description: str
    products: tuple
    arrival_rate: float
    weights: tuple


CONDITIONS = [
    Condition('familiar', '1. Normal day, familiar products P1-P4', FAMILIAR, ENV.arrival_rate, EVAL_WEIGHTS),
    Condition('unseen_p5', '2a. P1-P4 plus unseen P5 (inside the training range)', FAMILIAR + ('P5',), ENV.arrival_rate, EVAL_WEIGHTS),
    Condition('unseen_p6', '2b. P1-P4 plus unseen P6 (outside the training range)', FAMILIAR + ('P6',), ENV.arrival_rate, EVAL_WEIGHTS),
    Condition('busy', '3. Busy day, arrival rate doubled', FAMILIAR, 2 * ENV.arrival_rate, EVAL_WEIGHTS),
    Condition('high_damage', '4. Damage weight raised to the top of its range', FAMILIAR, ENV.arrival_rate, EVAL_WEIGHTS[:2] + (ENV.weight_high[2],)),
]

# A list means one policy trained with several seeds
policies = rule_baselines() | {
    'PPO': [load_ppo(RUN_DIR / 'ppo' / f'seed{s}' / 'best_model.zip') for s in SEEDS],
    'DQN': [load_dqn(RUN_DIR / 'dqn' / f'seed{s}' / 'best_model.pt') for s in SEEDS],
}

# One row per condition, policy, seed and shift
rows = []
for cond in CONDITIONS:
    cfg = replace(ENV, arrival_rate=cond.arrival_rate)
    shifts = make_instances(cfg, cond.products, N_TEST_SHIFTS, TEST_SEED)
    for name, runs in policies.items():
        for run, policy in enumerate(runs if isinstance(runs, list) else [runs]):
            for shift, metrics in enumerate(evaluate(policy, shifts, cfg, cond.weights)):
                rows.append({'condition': cond.name, 'policy': name, 'run': run, 'shift': shift} | metrics)

df = pd.DataFrame(rows)
df.to_csv(RESULTS_DIR / 'test_shifts.csv', index=False)
df.head()


## 17. Results tables for Part C

Mean ± standard deviation across shifts (and across seeds for the learned policies). Tardiness and travel are in seconds; `careful` counts careful-mode picks.


In [ ]:
ORDER = ['FCFS', 'SPT', 'LRF', 'EDF', 'NTF', 'Random', 'PPO', 'DQN']


def fmt(stats):
    # 'mean ± std' strings from a frame with mean and std columns
    return stats['mean'].map('{:.2f}'.format) + ' ± ' + stats['std'].map('{:.2f}'.format)


# Table 1: test 1 (normal day, familiar products), every metric
stats = df[df['condition'] == 'familiar'].groupby('policy')[METRICS].agg(['mean', 'std'])
familiar_table = pd.DataFrame({m: fmt(stats[m]) for m in METRICS}).reindex(ORDER).fillna('-')
familiar_table


In [ ]:
# Table 2: score under every test condition
stats = df.groupby(['policy', 'condition'])['score'].agg(['mean', 'std'])
score_table = fmt(stats).unstack('condition').reindex(index=ORDER, columns=[c.name for c in CONDITIONS]).fillna('-')
score_table


In [ ]:
# Save both tables as Markdown to paste into the Part C document
def markdown_lines(table):
    yield '| policy | ' + ' | '.join(table.columns) + ' |'
    yield '|---' * (len(table.columns) + 1) + '|'
    for name, row in zip(table.index, table.values):
        yield '| ' + name + ' | ' + ' | '.join(row) + ' |'


with open(RESULTS_DIR / 'partc_tables.md', 'w') as f:
    for title, table in [('Normal day, familiar products', familiar_table), ('Score by test condition', score_table)]:
        print('## ' + title, file=f)
        print(file=f)
        for line in markdown_lines(table):
            print(line, file=f)
        print(file=f)

print((RESULTS_DIR / 'partc_tables.md').read_text())


## 18. Changed penalties: does the policy adapt without retraining?

Test 4 raises the damage weight only through the observation. A policy that adapts should drop fewer items, mainly by picking in careful mode more often. Rules always pick in normal mode.


In [ ]:
# Mean drops, careful picks and score per shift, normal vs raised damage weight
(
    df[df['condition'].isin(['familiar', 'high_damage'])]
    .groupby(['policy', 'condition'])[['drops', 'careful', 'score']]
    .mean()
    .round(2)
    .unstack('condition')
    .reindex(ORDER)
)


## Results

- Key observations:
- Surprises or failure modes:
- Decision: continue, pivot, or stop:

## Next steps

- CP-SAT clairvoyant bound per shift, to measure how much room remains.
- Hyperparameter sweep: learning rate, gamma, epsilon decay (DQN) and entropy coefficient (PPO), several seeds each.
- Scale training to the Part B budget of 1M steps once the learning curves settle.
